# 03 - Gaussian Naive Bayes

Target: `DiagnosisType` (Normal / Non-GDM, Tipe 2, Gestasional), data bersih dari notebook 02.

Langkah:
1. Sisihkan 20% data sebagai data uji akhir (hold-out). Data ini baru dipakai sekali di notebook 04.
2. Seimbangkan kelas pada data latih. Data bersih tidak seimbang karena kelas Tipe 2 jauh lebih
   sedikit. Penyeimbangan hanya dilakukan pada data latih, tidak pada data uji, dan hasilnya tidak
   disimpan ke dataset.
3. Pada 80% data (data pengembangan), bandingkan cara penyeimbangan, transformasi fitur, prior
   kelas, dan ablasi fitur dengan repeated stratified 5-fold cross validation (10 kali ulang,
   total 50 fold).
4. Latih model final pada data latih yang sudah seimbang, periksa prior, mean, dan variansi, lalu
   hitung posterior secara manual.

Metrik utama adalah balanced accuracy dan F1 makro karena data uji tetap memakai proporsi kelas asli.

In [1]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

import json
import warnings

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from scipy import stats
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer

# kelas yang tidak muncul di subset (misalnya Gestasional di subset Pima) memicu peringatan
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS, N_REPEATS = 5, 10
ALPHA_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
KELAS = ["Normal / Non-GDM", "Tipe 2", "Gestasional"]
FITUR_AWAL = ["Age", "Pregnancies", "BMI", "BloodPressure", "Glucose"]

df = pd.read_csv(PROCESSED_DIR / "merged_diagnosis.csv")
assert df[FITUR_AWAL].notna().all().all() and not df.duplicated().any()
assert set(df["DiagnosisType"]) == set(KELAS)
print(f"Data bersih: {df.shape}")
print(df["DiagnosisType"].value_counts().reindex(KELAS).to_dict())

Data bersih: (3160, 7)
{'Normal / Non-GDM': 1539, 'Tipe 2': 249, 'Gestasional': 1372}


### Fungsi

In [2]:
class GaussianNBPrior(GaussianNB):
    """GaussianNB dengan prior campuran: (1 - alpha) x prior empiris + alpha x prior seragam.

    Prior dihitung dari data latih di dalam fit, jadi aman dipakai di dalam cross validation.
    alpha = 0 berarti prior empiris, alpha = 1 berarti prior seragam.
    """

    def __init__(self, alpha=0.0, var_smoothing=1e-9):
        super().__init__(var_smoothing=var_smoothing)
        self.alpha = alpha

    def fit(self, X, y, sample_weight=None):
        self.priors = prior_campuran(y, self.alpha)
        return super().fit(X, y, sample_weight=sample_weight)


def prior_campuran(y, alpha):
    _, jumlah = np.unique(np.asarray(y), return_counts=True)
    return (1 - alpha) * jumlah / jumlah.sum() + alpha / len(jumlah)


PENYEIMBANG = {
    "undersampling": lambda: RandomUnderSampler(random_state=RANDOM_STATE),
    "oversampling": lambda: RandomOverSampler(random_state=RANDOM_STATE),
    "SMOTE": lambda: SMOTE(random_state=RANDOM_STATE),
}


def buat_model(transformasi="tanpa", penyeimbangan="tanpa", alpha=0.0):
    """Pipeline: transformasi (opsional) -> penyeimbangan kelas (opsional) -> Gaussian NB.

    Langkah penyeimbangan hanya berjalan saat fit. Di dalam cross validation yang diseimbangkan
    hanya fold latih, sedangkan fold uji dan data uji akhir tidak pernah diubah.
    """
    langkah = []
    if transformasi == "log1p":
        langkah.append(("transformasi", FunctionTransformer(np.log1p)))
    elif transformasi == "yeo-johnson":
        langkah.append(("transformasi", PowerTransformer(method="yeo-johnson")))
    if penyeimbangan != "tanpa":
        langkah.append(("penyeimbang", PENYEIMBANG[penyeimbangan]()))
    langkah.append(("nb", GaussianNBPrior(alpha=alpha)))
    return ImbPipeline(langkah)


def skor_prediksi(y_benar, y_pred, sumber):
    """Metrik utama, metrik Tipe 2, dan balanced accuracy per sumber data."""
    y_benar, y_pred, sumber = np.asarray(y_benar), np.asarray(y_pred), np.asarray(sumber)
    hasil = {
        "balanced_accuracy": balanced_accuracy_score(y_benar, y_pred),
        "f1_makro": f1_score(y_benar, y_pred, average="macro"),
        "akurasi": accuracy_score(y_benar, y_pred),
        "recall_tipe2": recall_score(y_benar, y_pred, labels=["Tipe 2"], average="macro", zero_division=0),
        "f1_tipe2": f1_score(y_benar, y_pred, labels=["Tipe 2"], average="macro", zero_division=0),
    }
    for s in ["Pima", "GDM"]:
        bagian = sumber == s
        hasil[f"bal_acc_{s}"] = balanced_accuracy_score(y_benar[bagian], y_pred[bagian])
    return hasil


def cv_berulang(model, X, y, sumber):
    """Repeated stratified k-fold (5 x 10). Semua langkah ada di dalam model, jadi di-fit per fold."""
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    hasil = []
    for latih, uji in cv.split(X, y):
        m = clone(model).fit(X.iloc[latih], y.iloc[latih])
        hasil.append(skor_prediksi(y.iloc[uji], m.predict(X.iloc[uji]), sumber.iloc[uji]))
    return pd.DataFrame(hasil)


def ringkas_cv(daftar, kolom=("balanced_accuracy", "f1_makro", "akurasi", "recall_tipe2", "f1_tipe2",
                              "bal_acc_Pima", "bal_acc_GDM")):
    """Rata-rata dan simpangan baku skor 50 fold untuk tiap model."""
    tabel = {}
    for nama, skor in daftar.items():
        tabel[nama] = {k: f"{skor[k].mean():.4f} ± {skor[k].std():.4f}" for k in kolom}
    return pd.DataFrame(tabel).T


def uji_t_terkoreksi(skor_a, skor_b):
    """Uji t berpasangan terkoreksi (Nadeau dan Bengio, 2003) untuk repeated k-fold.

    Fold pada repeated CV saling tumpang tindih, jadi variansi selisih dikoreksi
    dengan faktor (1/k + n_uji/n_latih). Mengembalikan (rata-rata selisih, nilai p).
    """
    d = np.asarray(skor_a) - np.asarray(skor_b)
    k = len(d)
    rasio = 1 / (N_SPLITS - 1)
    t = d.mean() / np.sqrt((1 / k + rasio) * d.var(ddof=1))
    return d.mean(), 2 * stats.t.sf(abs(t), df=k - 1)


def log_pdf_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


def posterior_manual(x, nb):
    """P(kelas | x) = prior x perkalian P(x_i | kelas), dihitung dalam bentuk log."""
    log_likelihood = np.array([
        log_pdf_normal(np.asarray(x, dtype=float), nb.theta_[k], nb.var_[k]).sum()
        for k in range(len(nb.classes_))
    ])
    joint = np.log(nb.class_prior_) + log_likelihood
    probs = np.exp(joint - joint.max())
    return probs / probs.sum()

## 1. Data uji akhir (hold-out)

Pembagian 80:20 memakai stratify agar proporsi kelas sama. Data uji tidak disentuh sampai notebook 04
dan tidak ikut diseimbangkan.

In [3]:
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    df[FITUR_AWAL], df["DiagnosisType"], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["DiagnosisType"]
)
sumber_dev = df.loc[X_dev.index, "SumberData"]
print(f"Data pengembangan: {len(X_dev)} | data uji akhir: {len(X_holdout)}")
print("Tipe 2 di data uji akhir:", int((y_holdout == "Tipe 2").sum()))

Data pengembangan: 2528 | data uji akhir: 632
Tipe 2 di data uji akhir: 50


## 2. Kemiringan distribusi dan transformasi

Kemiringan dihitung pada data pengembangan, sebelum dan sesudah transformasi. Di dalam cross
validation, transformasi berada di dalam Pipeline sehingga hanya di-fit pada fold latih.

In [4]:
tabel_miring = pd.DataFrame({
    "sebelum": X_dev.skew(),
    "log1p": np.log1p(X_dev).skew(),
    "yeo-johnson": pd.DataFrame(PowerTransformer().fit_transform(X_dev), columns=FITUR_AWAL).skew(),
}).round(3)
tabel_miring

,sebelum,log1p,yeo-johnson
Age,0.855,0.185,0.014
Pregnancies,2.530,0.221,-0.021
BMI,0.611,-0.026,-0.000
BloodPressure,0.273,-0.469,0.029
Glucose,1.133,-0.014,0.000


## 3. Penyeimbangan kelas pada data latih

Tiga cara penyeimbangan dicoba:
- random undersampling: baris kelas besar dibuang secara acak sampai jumlahnya sama dengan kelas
  terkecil,
- random oversampling: baris kelas kecil diambil ulang secara acak sampai jumlahnya sama dengan
  kelas terbesar,
- SMOTE: baris baru untuk kelas kecil dibuat dari interpolasi antara pasien asli dan tetangga
  terdekatnya di kelas yang sama.

Ketiganya hanya dijalankan pada data latih, lewat Pipeline imblearn. Tabel berikut menunjukkan
jumlah tiap kelas pada data pengembangan sebelum dan sesudah diseimbangkan. Hasil penyeimbangan
tidak disimpan ke berkas data.

In [5]:
distribusi_seimbang = {"sebelum": y_dev.value_counts()}
for nama, buat in PENYEIMBANG.items():
    _, y_seimbang_contoh = buat().fit_resample(X_dev, y_dev)
    distribusi_seimbang[nama] = pd.Series(y_seimbang_contoh).value_counts()
pd.DataFrame(distribusi_seimbang).reindex(KELAS)

,sebelum,undersampling,oversampling,SMOTE
DiagnosisType,,,,
Normal / Non-GDM,1231,199,1231,1231
Tipe 2,199,199,1231,1231
Gestasional,1098,199,1231,1231


## 4. Penyeimbangan, transformasi, dan prior

Setiap transformasi (tanpa, log1p, Yeo-Johnson) dicoba dengan enam pilihan:
- tanpa penyeimbangan, prior empiris (proporsi kelas di data latih),
- tanpa penyeimbangan, prior seragam (1/3 untuk tiap kelas),
- tanpa penyeimbangan, prior tuning: alpha dipilih dengan GridSearchCV (F1 makro) di dalam setiap
  fold latih, jadi ini nested CV,
- random undersampling, random oversampling, dan SMOTE.

Pada data yang sudah seimbang, prior empiris otomatis menjadi 1/3 untuk tiap kelas. Bedanya dengan
prior seragam, mean dan variansi tiap kelas juga dihitung dari data yang sudah seimbang.

In [6]:
def model_tuning(transformasi):
    cv_dalam = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return GridSearchCV(buat_model(transformasi), {"nb__alpha": ALPHA_GRID}, scoring="f1_macro", cv=cv_dalam)


hasil_konfigurasi = {}
for transformasi in ["tanpa", "log1p", "yeo-johnson"]:
    hasil_konfigurasi[(transformasi, "prior empiris")] = cv_berulang(buat_model(transformasi), X_dev, y_dev, sumber_dev)
    hasil_konfigurasi[(transformasi, "prior seragam")] = cv_berulang(buat_model(transformasi, alpha=1.0), X_dev, y_dev, sumber_dev)
    hasil_konfigurasi[(transformasi, "prior tuning")] = cv_berulang(model_tuning(transformasi), X_dev, y_dev, sumber_dev)
    for nama in PENYEIMBANG:
        hasil_konfigurasi[(transformasi, nama)] = cv_berulang(buat_model(transformasi, nama), X_dev, y_dev, sumber_dev)

tabel_konfigurasi = ringkas_cv({f"{t} | {p}": s for (t, p), s in hasil_konfigurasi.items()})
tabel_konfigurasi

,balanced_accuracy,f1_makro,akurasi,recall_tipe2,f1_tipe2,bal_acc_Pima,bal_acc_GDM
tanpa | prior empiris,0.7718 ± 0.0275,0.7720 ± 0.0264,0.8654 ± 0.0154,0.5215 ± 0.0737,0.5252 ± 0.0646,0.6396 ± 0.0448,0.9201 ± 0.0149
tanpa | prior seragam,0.8248 ± 0.0258,0.7828 ± 0.0211,0.8540 ± 0.0153,0.7387 ± 0.0680,0.5767 ± 0.0445,0.6746 ± 0.0395,0.9109 ± 0.0153
tanpa | prior tuning,0.8145 ± 0.0283,0.7793 ± 0.0217,0.8547 ± 0.0155,0.7003 ± 0.0798,0.5645 ± 0.0461,0.6643 ± 0.0411,0.9125 ± 0.0157
tanpa | undersampling,0.8212 ± 0.0243,0.7797 ± 0.0203,0.8530 ± 0.0152,0.7277 ± 0.0672,0.5680 ± 0.0455,0.6681 ± 0.0419,0.9105 ± 0.0180
tanpa | oversampling,0.8250 ± 0.0249,0.7824 ± 0.0210,0.8541 ± 0.0156,0.7392 ± 0.0645,0.5746 ± 0.0431,0.6741 ± 0.0379,0.9114 ± 0.0156
tanpa | SMOTE,0.8286 ± 0.0240,0.7811 ± 0.0194,0.8523 ± 0.0149,0.7564 ± 0.0648,0.5724 ± 0.0391,0.6751 ± 0.0352,0.9101 ± 0.0157
log1p | prior empiris,0.7451 ± 0.0272,0.7487 ± 0.0275,0.8391 ± 0.0159,0.4924 ± 0.0710,0.5128 ± 0.0669,0.6419 ± 0.0464,0.8784 ± 0.0149
log1p | prior seragam,0.8273 ± 0.0179,0.7552 ± 0.0166,0.8191 ± 0.0151,0.8357 ± 0.0419,0.5589 ± 0.0293,0.6877 ± 0.0314,0.8662 ± 0.0163
log1p | prior tuning,0.7867 ± 0.0265,0.7608 ± 0.0217,0.8329 ± 0.0163,0.6567 ± 0.0836,0.5602 ± 0.0477,0.6726 ± 0.0410,0.8738 ± 0.0156
log1p | undersampling,0.8261 ± 0.0192,0.7548 ± 0.0180,0.8200 ± 0.0158,0.8287 ± 0.0424,0.5539 ± 0.0337,0.6845 ± 0.0387,0.8682 ± 0.0183


In [7]:
# model final wajib memakai data latih yang seimbang, jadi yang dipilih hanya konfigurasi dengan penyeimbangan
kandidat = {k: s for k, s in hasil_konfigurasi.items() if k[1] in PENYEIMBANG}
TRANSFORMASI, PENYEIMBANGAN = max(
    kandidat, key=lambda k: (round(kandidat[k]["f1_makro"].mean(), 3), kandidat[k]["balanced_accuracy"].mean())
)
skor_final_cv = kandidat[(TRANSFORMASI, PENYEIMBANGAN)]

print(f"Konfigurasi terpilih: transformasi={TRANSFORMASI}, penyeimbangan={PENYEIMBANGAN}")
print(f"F1 makro CV {skor_final_cv['f1_makro'].mean():.4f}, balanced accuracy CV {skor_final_cv['balanced_accuracy'].mean():.4f}")

for pembanding in ["prior empiris", "prior seragam"]:
    skor_banding = hasil_konfigurasi[(TRANSFORMASI, pembanding)]
    for metrik in ["f1_makro", "balanced_accuracy", "recall_tipe2"]:
        selisih, p = uji_t_terkoreksi(skor_final_cv[metrik], skor_banding[metrik])
        print(f"  dibanding tanpa penyeimbangan ({pembanding}), {metrik:18s}: selisih {selisih:+.4f}, p = {p:.4f}")

Konfigurasi terpilih: transformasi=tanpa, penyeimbangan=oversampling
F1 makro CV 0.7824, balanced accuracy CV 0.8250
  dibanding tanpa penyeimbangan (prior empiris), f1_makro          : selisih +0.0104, p = 0.3213
  dibanding tanpa penyeimbangan (prior empiris), balanced_accuracy : selisih +0.0532, p = 0.0000
  dibanding tanpa penyeimbangan (prior empiris), recall_tipe2      : selisih +0.2178, p = 0.0000
  dibanding tanpa penyeimbangan (prior seragam), f1_makro          : selisih -0.0004, p = 0.8550
  dibanding tanpa penyeimbangan (prior seragam), balanced_accuracy : selisih +0.0002, p = 0.9442
  dibanding tanpa penyeimbangan (prior seragam), recall_tipe2      : selisih +0.0005, p = 0.9465


Konfigurasi dipilih dari F1 makro rata-rata 50 fold (dibulatkan 3 angka, lalu balanced accuracy
sebagai penentu bila sama), hanya di antara konfigurasi yang menyeimbangkan data latih. Nilai p
memakai uji t terkoreksi, sehingga bisa dipakai untuk melihat apakah perbedaannya dengan model
tanpa penyeimbangan cukup meyakinkan.

## 5. Ablasi fitur

Setiap percobaan membuang satu fitur, lalu dibandingkan dengan model lima fitur memakai konfigurasi
terpilih. Perhatian khusus pada dua fitur:
- `Pregnancies`, karena artinya berbeda: di Pima jumlah kehamilan seumur hidup (0 sampai 17), di GDM
  kehamilan keberapa yang sedang dijalani (1 sampai 4).
- `Age`, karena berkorelasi 0,54 dengan `Pregnancies` di Pima, sehingga melanggar asumsi independensi.

Aturannya: `Pregnancies` dibuang jika penurunan balanced accuracy dan F1 makro sama-sama tidak
signifikan (p >= 0,05).

In [8]:
dasar = skor_final_cv
baris_ablasi = []
for fitur in FITUR_AWAL:
    sisa = [f for f in FITUR_AWAL if f != fitur]
    skor = cv_berulang(buat_model(TRANSFORMASI, PENYEIMBANGAN), X_dev[sisa], y_dev, sumber_dev)
    baris = {"dibuang": fitur}
    for metrik in ["balanced_accuracy", "f1_makro", "bal_acc_Pima", "bal_acc_GDM"]:
        selisih, p = uji_t_terkoreksi(skor[metrik], dasar[metrik])
        baris[metrik] = round(skor[metrik].mean(), 4)
        baris[f"selisih_{metrik}"] = round(selisih, 4)
        baris[f"p_{metrik}"] = round(p, 4)
    baris_ablasi.append(baris)

tabel_ablasi = pd.DataFrame(baris_ablasi).set_index("dibuang")
print("Model lima fitur:", {k: round(dasar[k].mean(), 4) for k in ["balanced_accuracy", "f1_makro", "bal_acc_Pima", "bal_acc_GDM"]})
tabel_ablasi[["balanced_accuracy", "selisih_balanced_accuracy", "p_balanced_accuracy",
              "f1_makro", "selisih_f1_makro", "p_f1_makro", "bal_acc_Pima", "bal_acc_GDM"]]

Model lima fitur: {'balanced_accuracy': np.float64(0.825), 'f1_makro': np.float64(0.7824), 'bal_acc_Pima': np.float64(0.6741), 'bal_acc_GDM': np.float64(0.9114)}


,balanced_accuracy,selisih_balanced_accuracy,p_balanced_accuracy,f1_makro,selisih_f1_makro,p_f1_makro,bal_acc_Pima,bal_acc_GDM
dibuang,,,,,,,,
Age,0.8253,0.0004,0.9727,0.7803,-0.0021,0.8128,0.6400,0.9391
Pregnancies,0.8032,-0.0218,0.0728,0.7508,-0.0316,0.0005,0.6463,0.9032
BMI,0.7641,-0.0608,0.0002,0.7530,-0.0294,0.0240,0.6430,0.8844
BloodPressure,0.8175,-0.0075,0.2987,0.7724,-0.0100,0.0636,0.6731,0.8979
Glucose,0.7550,-0.0700,0.0000,0.7062,-0.0762,0.0000,0.6348,0.8061


In [9]:
p_hamil = tabel_ablasi.loc["Pregnancies", ["p_balanced_accuracy", "p_f1_makro"]]
buang_pregnancies = bool((p_hamil >= 0.05).all())
FITUR = [f for f in FITUR_AWAL if not (f == "Pregnancies" and buang_pregnancies)]
print("Nilai p tanpa Pregnancies:", p_hamil.to_dict())
print("Pregnancies dibuang:", buang_pregnancies, "| fitur dipakai:", FITUR)

Nilai p tanpa Pregnancies: {'p_balanced_accuracy': 0.0728, 'p_f1_makro': 0.0005}
Pregnancies dibuang: False | fitur dipakai: ['Age', 'Pregnancies', 'BMI', 'BloodPressure', 'Glucose']


Hasil ablasi dibahas di README. Fitur yang tidak signifikan tetap dipertahankan kecuali `Pregnancies`
memenuhi aturan di atas, karena tidak ada fitur pengganti dari sumber lain.

## 6. Model final

Model final dilatih pada seluruh data pengembangan setelah diseimbangkan. Berkas model yang disimpan
hanya berisi langkah yang dipakai saat prediksi (transformasi bila ada dan `GaussianNB` dengan prior
eksplisit), karena penyeimbangan tidak berperan saat prediksi. Dengan begitu aplikasi tidak
bergantung pada kelas buatan di notebook ini.

In [10]:
X_latih = X_dev[FITUR]
model_latih = buat_model(TRANSFORMASI, PENYEIMBANGAN).fit(X_latih, y_dev)
nb_latih = model_latih.named_steps["nb"]

# ulangi penyeimbangan dengan random_state yang sama untuk melihat data latih yang dipakai model
masukan_latih = model_latih.named_steps["transformasi"].transform(X_latih) if TRANSFORMASI != "tanpa" else X_latih
X_seimbang, y_seimbang = clone(model_latih.named_steps["penyeimbang"]).fit_resample(masukan_latih, y_dev)

langkah = [("transformasi", model_latih.named_steps["transformasi"])] if TRANSFORMASI != "tanpa" else []
langkah.append(("nb", GaussianNB(priors=nb_latih.class_prior_).fit(X_seimbang, y_seimbang)))
model = Pipeline(langkah)
nb = model.named_steps["nb"]
assert np.allclose(nb.theta_, nb_latih.theta_) and np.allclose(nb.var_, nb_latih.var_)
assert (model.predict(X_latih) == model_latih.predict(X_latih)).all()

tabel_latih = pd.DataFrame({
    "jumlah_sebelum": y_dev.value_counts(),
    "jumlah_sesudah": pd.Series(y_seimbang).value_counts(),
    "prior_empiris_data_asli": y_dev.value_counts(normalize=True),
    "prior_dipakai": pd.Series(nb.class_prior_, index=nb.classes_),
}).reindex(KELAS).round(4)
print(f"Data latih: {len(y_dev)} baris asli -> {len(y_seimbang)} baris setelah {PENYEIMBANGAN}")
tabel_latih

Data latih: 2528 baris asli -> 3693 baris setelah oversampling


,jumlah_sebelum,jumlah_sesudah,prior_empiris_data_asli,prior_dipakai
Normal / Non-GDM,1231,1231,0.4869,0.3333
Tipe 2,199,1231,0.0787,0.3333
Gestasional,1098,1231,0.4343,0.3333


## 7. Mean dan variansi fitur di tiap kelas (data latih seimbang)

In [11]:
nama_kolom = FITUR if TRANSFORMASI == "tanpa" else [f"{f} (setelah transformasi)" for f in FITUR]
distribusi = pd.DataFrame({f"mean_{k}": nb.theta_[i] for i, k in enumerate(nb.classes_)}, index=nama_kolom)
for i, k in enumerate(nb.classes_):
    distribusi[f"var_{k}"] = nb.var_[i]
distribusi.round(2)

,mean_Gestasional,mean_Normal / Non-GDM,mean_Tipe 2,var_Gestasional,var_Normal / Non-GDM,var_Tipe 2
Age,36.10,29.78,37.19,35.55,64.84,131.25
Pregnancies,2.31,2.45,4.87,0.78,3.58,13.60
BMI,31.09,25.96,34.95,19.69,37.44,38.09
BloodPressure,90.23,74.37,74.40,115.95,108.25,129.49
Glucose,206.70,129.86,143.18,2094.35,662.74,888.46


## 8. Hitung posterior secara manual

$$P(y \mid x_1, \dots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

Supaya tidak terjadi underflow, perhitungan dilakukan dalam bentuk log. Hasilnya dibandingkan dengan
`predict_proba()` untuk satu sampel, lalu untuk seluruh data pengembangan asli.

In [12]:
def masukan_nb(X):
    """Nilai yang masuk ke GaussianNB (sesudah transformasi jika ada)."""
    return model[:-1].transform(X) if len(model) > 1 else X.to_numpy(dtype=float)


sampel = X_latih.iloc[[0]]
p_manual = posterior_manual(masukan_nb(sampel)[0], nb)
p_sklearn = model.predict_proba(sampel)[0]
print(sampel.to_string(index=False))
print(f"Label sebenarnya: {y_dev.iloc[0]}")
for kelas, a, b in zip(nb.classes_, p_manual, p_sklearn):
    print(f"  {kelas:<17} manual {a:.6f}   sklearn {b:.6f}")
assert np.allclose(p_manual, p_sklearn, atol=1e-6)

semua_manual = np.array([posterior_manual(baris, nb) for baris in masukan_nb(X_latih)])
assert np.allclose(semua_manual, model.predict_proba(X_latih), atol=1e-6)
print(f"Posterior manual sama dengan predict_proba() untuk seluruh {len(X_latih)} data pengembangan.")

 Age  Pregnancies  BMI  BloodPressure  Glucose
  27            2 20.4           70.0     99.0
Label sebenarnya: Normal / Non-GDM
  Gestasional       manual 0.001371   sklearn 0.001371
  Normal / Non-GDM  manual 0.988795   sklearn 0.988795
  Tipe 2            manual 0.009834   sklearn 0.009834
Posterior manual sama dengan predict_proba() untuk seluruh 2528 data pengembangan.


## 9. Simpan model

`info_model.json` berisi fitur, cara penyeimbangan, prior, jumlah data latih sebelum dan sesudah
diseimbangkan, dan rentang nilai data latih. Rentang dihitung dari pasien asli saja (bukan baris
hasil penyeimbangan) dan dipisah menurut kondisi: pasien GDM dianggap kondisi hamil, pasien Pima
dianggap tidak hamil. Aplikasi memakai rentang ini untuk memberi peringatan bila input berada di
luar data latih.

In [13]:
def rentang(bagian):
    return {f: [float(bagian[f].min()), float(bagian[f].max())] for f in FITUR}


info = {
    "fitur": FITUR,
    "kelas": nb.classes_.tolist(),
    "transformasi": TRANSFORMASI,
    "penyeimbangan": PENYEIMBANGAN,
    "prior": dict(zip(nb.classes_.tolist(), [round(float(p), 6) for p in nb.class_prior_])),
    "jumlah_data": {"bersih": len(df), "pengembangan": len(X_dev), "uji_akhir": len(X_holdout),
                    "latih_setelah_seimbang": len(y_seimbang)},
    "distribusi_kelas_latih": {
        "sebelum": {k: int(v) for k, v in y_dev.value_counts().reindex(KELAS).items()},
        "sesudah": {k: int(v) for k, v in pd.Series(y_seimbang).value_counts().reindex(KELAS).items()},
    },
    "rentang_input": {
        "hamil": rentang(X_latih[sumber_dev == "GDM"]),
        "tidak_hamil": rentang(X_latih[sumber_dev == "Pima"]),
    },
}

joblib.dump(model, MODELS_DIR / "gnb_diagnosis.joblib")
with open(MODELS_DIR / "info_model.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("Tersimpan: models/gnb_diagnosis.joblib, models/info_model.json")
print(json.dumps(info["distribusi_kelas_latih"], indent=1, ensure_ascii=False))

Tersimpan: models/gnb_diagnosis.joblib, models/info_model.json
{
 "sebelum": {
  "Normal / Non-GDM": 1231,
  "Tipe 2": 199,
  "Gestasional": 1098
 },
 "sesudah": {
  "Normal / Non-GDM": 1231,
  "Tipe 2": 1231,
  "Gestasional": 1231
 }
}
